In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence

In [2]:
import numpy as np
import random
from tqdm import tqdm
import os
from collections import Counter
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [3]:
class Config:
    SEED=42
    MAX_VOCAB_SIZE = 25000
    MAX_LEN = 256
    BATCH_SIZE = 64
    EMBEDDING_DIM = 128
    HIDDEN_DIM = 256
    NUM_LAYERS = 2
    DROPOUT = 0.5
    LR=0.01
    WEIGHT_DECAY=1e-5
    EPOCHS=20
    GRAD_CLIP=1.0
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH='best_RNN_sentiment_model.pth'

# Set seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)

In [4]:
# Data Pre-Processing
def clean_text(text):
    text = re.sub(r'<.*>','',text)
    text = re.sub(r'[^a-zA-Z\s]', '',text)
    text = text.lower().strip()
    return text

In [5]:
class IMDBDataset(Dataset):
    def __init__(self,texts,labels,vocab=None,max_len=256):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        tokens = [self.vocab.get(word,self.vocab['<unk>']) for word in text.split()][:self.max_len]
        return torch.tensor(tokens, dtype=torch.long), torch.tensor(label, dtype=torch.long)

In [6]:
def build_vocab(texts,max_size=25000):
    word_freq = Counter()
    for text in texts:
        word_freq.update(text.split())
    
    vocab = {'<pad>':0 , '<unk>':1}
    for word, freq in word_freq.most_common(max_size-2):
        vocab[word] = len(vocab)

    print(f'Vocabulary size: {len(vocab)}')
    return vocab

In [7]:
def collate_fn(batch):
    texts, labels = zip(*batch)
    texts_padded = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.stack(labels)
    return texts_padded, labels

In [8]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout=0.5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers>1 else 0,
            bidirectional=False
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim,1)

    def forward(self,x):
        embedded = self.embedding(x)
        embedded = self.dropout(embedded)
        output, hidden = self.rnn(embedded)

        # Take the last layers's last hidden state
        hidden = hidden[-1]
        hidden = self.dropout(hidden)

        out = self.fc(hidden)
        return out.squeeze(1)

In [9]:
def train_epoch(model, dataloader,optimizer,criterion,device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for texts, labels in tqdm(dataloader, desc='Training'):
        texts, labels = texts.to(device), labels.to(device).float()

        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()

        preds = (torch.sigmoid(outputs)>0.5).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return avg_loss, acc, f1
    

In [10]:
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels= [], []

    with torch.no_grad():
        for texts, labels in tqdm(dataloader, desc='Evaluating'):
            texts, labels = texts.to(device), labels.to(device).float()
            outputs = model(texts)
            loss = criterion(outputs, labels)

            total_loss+=loss.item()

            preds  = (torch.sigmoid(outputs)>0.5).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels,all_preds)
    return avg_loss, acc, f1

In [11]:
def predict_sentiment(model,text,vocab,device,max_len=256):
    model.eval()
    cleaned = clean_text(text)
    tokens = [vocab.get(word, vocab['<unk>']) for word in cleaned.split()][:max_len]
    tensor = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(tensor)
        prob = torch.sigmoid(output).item()

    sentiment = 'Positive' if prob > 0.5 else 'Negative'
    return sentiment, prob

In [12]:
from datasets import load_dataset

def main():
    print(f'Using Device : {Config.DEVICE}')
    dataset = load_dataset('imdb')
    print('Loading IMDB dataset......')

    train_iter = dataset['train']
    test_iter = dataset['test']

    train_texts, train_labels= [],[]
    for item in train_iter:
        cleaned = clean_text(item['text'])
        train_texts.append(cleaned)
        train_labels.append(item['label'])

    test_texts, test_labels=[], []
    for item in test_iter:
        cleaned = clean_text(item['text'])
        test_texts.append(cleaned)
        test_labels.append(item['label'])

    print(f'Train samples: {len(train_texts)}, Test samples: {len(test_texts)}')

    print('Building Vocabulary......')
    vocab = build_vocab(train_texts, Config.MAX_VOCAB_SIZE)

    train_dataset = IMDBDataset(train_texts, train_labels, vocab, Config.MAX_LEN)
    test_dataset = IMDBDataset(test_texts, test_labels, vocab, Config.MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE,shuffle=True, collate_fn=collate_fn, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    vocab_size=len(vocab)
    model = SimpleRNN(
        vocab_size=vocab_size,
        embedding_dim=Config.HIDDEN_DIM,
        hidden_dim=Config.HIDDEN_DIM,
        num_layers=Config.NUM_LAYERS,
        dropout = Config.DROPOUT
    ).to(Config.DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(),lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

    best_val_acc = 0.0
    print('\n======== Starting Training ========\n')

    for epoch in range(Config.EPOCHS):
        train_loss, train_acc, train_f1 = train_epoch(model,train_loader,optimizer,criterion, Config.DEVICE)
        val_loss, val_acc, val_f1 = evaluate(model, test_loader, criterion, Config.DEVICE)


        print(f'Epoch {epoch+1}/{Config.EPOCHS}')
        print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1:{train_f1:.4f}")
        print(f'Val Loss: {val_loss:.4f} | Acc:{val_acc:.4f} | F1: {val_f1:.4f}')

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'model_state_dict': model.state_dict(),
                'vocab': vocab,
                'Config' : {k: v for k,v in Config.__dict__.items() if not k.startswith('__')}

            },Config.MODEL_PATH)
            print(f' Best Model Saved! Val Acc: {val_acc:.4f}')
            print('-'*60)

    print(f'\nTraining completed! Best Validation Accuracy: {best_val_acc:.4f}')

    print('\n=== Inference Examples ===')
    checkpoint = torch.load(Config.MODEL_PATH,weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])

    test_reviews = [
        "This movie was absolutely fantastic! The acting and story were brilliant.",
        "Worst movie I have ever seen. Complete waste of time.",
        "It was okay, not great but not terrible either."
    ]

    for review in test_reviews:
        sentiment, confidence = predict_sentiment(model,review, vocab, Config.DEVICE)
        print(f'Review: {review[:100]}...')
        print(f'Prediction: {sentiment} ({confidence:.4f})\n')



In [13]:
if __name__ == '__main__':
    main()

Using Device : cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

...

Epoch 18/20
Train Loss: 0.7504 | Acc: 0.4974 | F1:0.4962
Val Loss: 0.7120 | Acc:0.5020 | F1: 0.6196


Evaluating: 100%|██████████| 391/391 [00:03<00:00, 104.03it/s]


Epoch 19/20
Train Loss: 0.7473 | Acc: 0.5001 | F1:0.5032
Val Loss: 0.6983 | Acc:0.5093 | F1: 0.5119


Evaluating: 100%|██████████| 391/391 [00:03<00:00, 103.44it/s]


Epoch 20/20
Train Loss: 0.7522 | Acc: 0.4980 | F1:0.4966
Val Loss: 0.7507 | Acc:0.5019 | F1: 0.5827

Training completed! Best Validation Accuracy: 0.5100

=== Inference Examples ===
Review: This movie was absolutely fantastic! The acting and story were brilliant....
Prediction: Negative (0.4007)

Review: Worst movie I have ever seen. Complete waste of time....
Prediction: Negative (0.3878)

Review: It was okay, not great but not terrible either....
Prediction: Positive (0.5707)



In [14]:
from google.colab import runtime

In [15]:
runtime.unassign()